## Task 1

In [3]:
import chess
from queue import PriorityQueue

def evaluate(board):
    piece_values = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9
    }

    score = 0
    for piece, value in piece_values.items():
        score += len(board.pieces(piece, chess.WHITE)) * value
        score -= len(board.pieces(piece, chess.BLACK)) * value

    return score


def beam_search(board, beam_width, depth_limit):

    pq = PriorityQueue()
    counter = 0

    pq.put((-evaluate(board), counter, board, []))

    depth = 0

    while not pq.empty() and depth < depth_limit:

        candidates = []

        while not pq.empty():
            score, _, state, path = pq.get()

            for move in state.legal_moves:
                new_board = state.copy()
                new_board.push(move)

                counter += 1
                new_score = -evaluate(new_board)

                candidates.append((new_score, counter, new_board, path + [move]))

        candidates.sort(key=lambda x: x[0])
        candidates = candidates[:beam_width]

        for item in candidates:
            pq.put(item)

        depth += 1

    best_score, _, best_board, best_moves = pq.get()

    return best_moves, -best_score


board = chess.Board()

moves, score = beam_search(board, beam_width=3, depth_limit=2)

print("Best move sequence:")
for m in moves:
    print(m)

print("Evaluation score:", score)


Best move sequence:
g1h3
g8h6
Evaluation score: 0


## Task 2

In [2]:
from math import sqrt

delivery_points = [ (2, 3), (5, 8), (1, 1), (7, 4), (6, 9) ]

# euclidean distance as heuristic fn
def heuristic(a, b):
    return sqrt((b[0] - a[0])**2 + (b[1] - a[1])**2)

# main fn
# no gpt use in this
def hill_climbing_delivery(points, start=(0, 0)):
    path = []
    visited = set()
    curr = start

    while curr:
        visited.add(curr)
        path.append(curr)

        if len(visited) == len(points) + 1:
            break

        best_point = None
        best_h = float('inf')

        for point in points:
            if point not in visited and heuristic(curr, point) < best_h:
                best_h = heuristic(curr, point)
                best_point = point

        if best_point is None:
            break

        curr = best_point

    total_dist = sum(heuristic(path[i], path[i+1]) for i in range(len(path) - 1))
    return path, total_dist

route, dist = hill_climbing_delivery(delivery_points)
print("Route:", route)
print(f"Total distance: {dist:.2f}")

Route: [(0, 0), (1, 1), (2, 3), (7, 4), (5, 8), (6, 9)]
Total distance: 14.64


## Task 3

In [1]:
def create_initial_population(no_of_cities, size_of_population):
    population = []
    route = list(range(no_of_cities))

    for _ in range(size_of_population):
        shuffled = route[1:]
        random.shuffle(shuffled)
        population.append([0] + shuffled)
    return population

In [3]:
import math
import random

def generate_cities(n):
    return [(random.randint(0, 100), random.randint(0, 100)) for _ in range(n)]

# this function is gpt
def build_dist_matrix(cities):
    n = len(cities)
    dist_matrix = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            dx = cities[i][0] - cities[j][0]
            dy = cities[i][1] - cities[j][1]
            dist_matrix[i][j] = math.sqrt(dx**2 + dy**2)
    return dist_matrix

# Use these throughout your GA
cities = generate_cities(10)
dist_matrix = build_dist_matrix(cities)

In [4]:
def total_distance(route, dist_matrix):
    total = 0
    n = len(route)
    for i in range(n):
        total += dist_matrix[route[i]][route[(i+1) % n]]
    return total

def fitness(route, dist_matrix):
    return 1 / total_distance(route, dist_matrix)

In [5]:
def get_parents(population, dist_matrix):
    random.shuffle(population)
    parents = []
    mid = len(population) // 2
    for i in range(mid):
        if fitness(population[i], dist_matrix) > fitness(population[i + mid], dist_matrix):
            parents.append(population[i])
        else:
            parents.append(population[i + mid])
    return parents

In [6]:
def ox(primary, secondary):
    start = random.randint(0, len(primary) - 1)
    end = random.randint(start, len(primary))
    child = primary[start:end]
    for i in secondary:
        if i not in child:
            child.append(i)
    return child

def apply_crossover(population):
    random.shuffle(population)
    children = []
    mid = len(population) // 2
    for i in range(mid):
        p1, p2 = population[i], population[i + mid]
        child1 = ox(p1, p2)
        child2 = ox(p2, p1)
        children.extend([child1, child2])
    return children

In [7]:
def apply_mutation(parents):
    
    for route in parents:
        if random.randint(1, 100) < 5:
            index1 = random.randint(1, len(route) - 1)
            index2 = random.randint(1, len(route) - 1)
            route[index1], route[index2] = route[index2], route[index1]
            
    return parents

In [8]:
def generate_new_population(population, dist_matrix):
    population_size = len(population)
    parents = get_parents(population, dist_matrix)
    offspring = apply_crossover(parents)
    offspring = apply_mutation(offspring)
    
    while len(offspring) < population_size:
        offspring.append(random.choice(population))
    
    return offspring[:population_size]

In [9]:
def get_best_route(population, dist_matrix):
    best = population[0]
    for route in population:
        if total_distance(route, dist_matrix) < total_distance(best, dist_matrix):
            best = route
    return best

# this function is also gpt
def run_ga(no_of_cities, population_size, generations):
    # Setup
    cities = generate_cities(no_of_cities)
    dist_matrix = build_dist_matrix(cities)
    population = create_initial_population(no_of_cities, population_size)
    
    best_route = get_best_route(population, dist_matrix)
    
    # Main loop
    for gen in range(generations):
        population = generate_new_population(population, dist_matrix)
        
        current_best = get_best_route(population, dist_matrix)
        if total_distance(current_best, dist_matrix) < total_distance(best_route, dist_matrix):
            best_route = current_best
        
        print(f"Generation {gen + 1} | Best Distance: {total_distance(best_route, dist_matrix):.2f}")
    
    return best_route, cities, dist_matrix

In [11]:
best_route, cities, dist_matrix = run_ga(10, 50, 100)
print(f"\nBest Route: {best_route}")

Generation 1 | Best Distance: 459.55
Generation 2 | Best Distance: 444.33
Generation 3 | Best Distance: 444.33
Generation 4 | Best Distance: 400.53
Generation 5 | Best Distance: 369.52
Generation 6 | Best Distance: 369.52
Generation 7 | Best Distance: 369.52
Generation 8 | Best Distance: 369.52
Generation 9 | Best Distance: 369.52
Generation 10 | Best Distance: 369.52
Generation 11 | Best Distance: 369.52
Generation 12 | Best Distance: 369.52
Generation 13 | Best Distance: 369.52
Generation 14 | Best Distance: 369.52
Generation 15 | Best Distance: 369.52
Generation 16 | Best Distance: 369.52
Generation 17 | Best Distance: 349.85
Generation 18 | Best Distance: 349.85
Generation 19 | Best Distance: 349.85
Generation 20 | Best Distance: 349.85
Generation 21 | Best Distance: 349.85
Generation 22 | Best Distance: 349.85
Generation 23 | Best Distance: 349.85
Generation 24 | Best Distance: 349.85
Generation 25 | Best Distance: 349.85
Generation 26 | Best Distance: 349.85
Generation 27 | Best 

## Task 4

In [4]:
def beam_search_task_allocation(tasks, processors, beam_width):

    initial_state = {
        "assign": {p: [] for p in processors},
        "load": {p: 0 for p in processors},
        "task_index": 0
    }

    beam = [initial_state]

    while beam:

        candidates = []

        for state in beam:

            i = state["task_index"]

            if i == len(tasks):
                return state

            task = tasks[i]

            for p in processors:

                new_assign = {k: v[:] for k, v in state["assign"].items()}
                new_load = state["load"].copy()

                new_assign[p].append(task["id"])
                new_load[p] += task["time"]

                candidates.append({
                    "assign": new_assign,
                    "load": new_load,
                    "task_index": i + 1
                })

        # heuristic = max processor load
        candidates.sort(key=lambda x: max(x["load"].values()))

        beam = candidates[:beam_width]

    return None


In [5]:
processors = ["P1", "P2"]

tasks = [
    {"id": "T1", "time": 4, "priority": 3},
    {"id": "T2", "time": 2, "priority": 2},
    {"id": "T3", "time": 6, "priority": 1},
]

result = beam_search_task_allocation(tasks, processors, beam_width=2)

print("Task Allocation:")
for p, jobs in result["assign"].items():
    print(p, ":", jobs)

print("Loads:", result["load"])


Task Allocation:
P1 : ['T1']
P2 : ['T2', 'T3']
Loads: {'P1': 4, 'P2': 8}
